# Smart Personal AI Assistant


This notebook implements a multi-agent personal assistant (Calendar + Gmail + Knowledge Base)
built on LangGraph's **Functional API** (`@task` / `@entrypoint`), with:

- Retrieval-Augmented Generation (RAG) over a small knowledge base
- A **Routing** pattern supervisor that dispatches to Calendar / Email / Knowledge agents
- Real human-in-the-loop approval via `interrupt()` / `Command(resume=...)`
- Short-term memory (`MemorySaver`, per-thread) **and** long-term memory (`InMemoryStore`, per-user)
- LangSmith tracing
- Error handling: a `RetryPolicy` on one task, and one genuine uncaught failure path

# ==========================================
# Module 1 — Setup & Installation
# ==========================================

In [1]:
!pip install -q \
langgraph \
langchain \
langchain-community \
langchain-openai \
langchain-google-genai \
langchain-huggingface \
sentence-transformers \
langsmith \
google-api-python-client \
google-auth \
google-auth-oauthlib \
google-auth-httplib2 \
faiss-cpu \
pypdf \
tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [45]:
from importlib.metadata import version

print("Langgraph:", version("langgraph"))
print("Langchain:", version("langchain"))
print("Langsmith:", version("langsmith"))
print("All packages imported successfully!")

Langgraph: 1.2.6
Langchain: 1.3.11
Langsmith: 0.9.1
All packages imported successfully!


In [4]:
import os

folders = [
    "data",
    "credentials",
    "vector_db",
    "outputs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


## 1.1 API Keys & LangSmith tracing

This section loads the Gemini API key plus a LangSmith API key from Colab Secrets, and turns on
LangSmith tracing for this project. It sets **both** `LANGCHAIN_API_KEY` and
`LANGSMITH_API_KEY` (LangChain's tracer looks for `LANGCHAIN_API_KEY`; the `langsmith` SDK
looks for `LANGSMITH_API_KEY` — setting both avoids version-dependent surprises), then the
next cell confirms authentication actually succeeds before any agent runs.

In [6]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# --- LangSmith / tracing ---
langsmith_key = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = langsmith_key
os.environ["LANGSMITH_API_KEY"] = langsmith_key

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Smart Personal Assistant"

print("LangSmith tracing enabled for project:", os.environ["LANGCHAIN_PROJECT"])

LangSmith tracing enabled for project: Smart Personal Assistant


In [7]:
# Sanity check that the LangSmith key actually authenticates
# (this replaces silently getting 401s on every trace call).
from langsmith import Client

client = Client()
try:
    # list_runs on a fresh project just needs auth to succeed, not return results
    list(client.list_runs(project_name=os.environ["LANGCHAIN_PROJECT"], limit=1))
    print("LangSmith authenticated successfully — tracing is live.")
except Exception as e:
    print("LangSmith auth still failing, check your LANGSMITH_API_KEY secret:", e)

LangSmith authenticated successfully — tracing is live.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Test it
response = llm.invoke("Hello! Tell me who you are in one sentence.")
print(response.content)

I am a large language model, trained by Google.


# ==========================================
# Module 2 — Google OAuth (Calendar + Gmail)
# ==========================================
Upload your OAuth `client_secret_*.json` (from Google Cloud Console) and complete the standard
Colab OAuth flow to produce `credentials/token.json`. This part of the notebook is unchanged
from the original working version — it was already functioning correctly.

In [9]:
import shutil

# Find the uploaded client-secret JSON file and move it into credentials/
for file in os.listdir():
    if file.endswith(".json") and "token" not in file:
        shutil.move(file, os.path.join("credentials", file))
        break

print("Client secret saved to credentials/")

Client secret saved to credentials/


In [11]:
# --- Run the OAuth flow once to produce credentials/token.json ---
# (Uses google-auth-oauthlib's InstalledAppFlow; run interactively the first time,
# then token.json is reused on subsequent runs.)
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials
import glob

SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.modify"
]

token_path = "credentials/token.json"

if os.path.exists(token_path):
    creds = Credentials.from_authorized_user_file(token_path, SCOPES)
else:
    secret_files = [f for f in glob.glob("credentials/*.json") if "token" not in f]
    flow = InstalledAppFlow.from_client_secrets_file(secret_files[0], SCOPES)
    creds = flow.run_console()
    with open(token_path, "w") as f:
        f.write(creds.to_json())

print("Credentials loaded successfully!")
print("Valid:", creds.valid)

Credentials loaded successfully!
Valid: False


In [12]:
from googleapiclient.discovery import build

calendar_service = build("calendar", "v3", credentials=creds)
gmail_service = build("gmail", "v1", credentials=creds)

print("Connected to Google Calendar and Gmail!")

Connected to Google Calendar and Gmail!


# ==========================================
# Module 3 — Calendar Functions
# ==========================================

In [13]:
from datetime import datetime, timedelta, timezone

def list_today_events():
    """Returns all events scheduled for today from the user's primary Google Calendar."""
    try:
        now = datetime.now(timezone.utc)
        end = now + timedelta(days=1)

        events_result = calendar_service.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            singleEvents=True,
            orderBy="startTime"
        ).execute()

        events = events_result.get("items", [])
        if not events:
            return "No events found for today."

        output = []
        for event in events:
            start = event["start"].get("dateTime", event["start"].get("date"))
            title = event.get("summary", "Untitled Event")
            output.append(f"{title}\n{start}\n")

        return "\n".join(output)

    except Exception as e:
        return f"Error: {e}"


In [14]:
from datetime import datetime, timedelta

def create_calendar_event(title, start_datetime, end_datetime=None, description="", location=""):
    """Creates a new event in the user's Google Calendar. Defaults to a 1-hour meeting."""
    try:
        if end_datetime is None:
            start = datetime.fromisoformat(start_datetime)
            end = start + timedelta(hours=1)
            end_datetime = end.isoformat()

        event = {
            "summary": title,
            "location": location,
            "description": description,
            "start": {"dateTime": start_datetime, "timeZone": "Asia/Riyadh"},
            "end": {"dateTime": end_datetime, "timeZone": "Asia/Riyadh"}
        }

        created_event = calendar_service.events().insert(calendarId="primary", body=event).execute()

        return {
            "status": "success",
            "message": "Event created successfully.",
            "event_id": created_event["id"],
            "link": created_event["htmlLink"]
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}


In [15]:
from datetime import datetime, timedelta, timezone

def search_calendar_events(keyword, days=30):
    """Search for calendar events containing a keyword."""
    try:
        now = datetime.now(timezone.utc)
        end = now + timedelta(days=days)

        events_result = calendar_service.events().list(
            calendarId="primary",
            timeMin=now.isoformat(),
            timeMax=end.isoformat(),
            q=keyword,
            singleEvents=True,
            orderBy="startTime"
        ).execute()

        events = events_result.get("items", [])
        if not events:
            return {"status": "success", "count": 0, "events": []}

        result = []
        for event in events:
            result.append({
                "id": event["id"],
                "title": event.get("summary", "Untitled"),
                "start": event["start"].get("dateTime", event["start"].get("date")),
                "link": event.get("htmlLink", "")
            })

        return {"status": "success", "count": len(result), "events": result}

    except Exception as e:
        return {"status": "error", "message": str(e)}


In [16]:
def update_calendar_event(event_id, title=None, start_datetime=None, end_datetime=None,
                           description=None, location=None):
    """Update an existing Google Calendar event."""
    try:
        event = calendar_service.events().get(calendarId="primary", eventId=event_id).execute()

        if title is not None:
            event["summary"] = title
        if description is not None:
            event["description"] = description
        if location is not None:
            event["location"] = location
        if start_datetime is not None:
            event["start"] = {"dateTime": start_datetime, "timeZone": "Asia/Riyadh"}
        if end_datetime is not None:
            event["end"] = {"dateTime": end_datetime, "timeZone": "Asia/Riyadh"}

        updated_event = calendar_service.events().update(
            calendarId="primary", eventId=event_id, body=event
        ).execute()

        return {
            "status": "success",
            "message": "Event updated successfully.",
            "event_id": updated_event["id"],
            "link": updated_event["htmlLink"]
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}


In [17]:
def delete_calendar_event(event_id):
    """Actually deletes an event from Google Calendar. Only ever called AFTER human
    confirmation — see Module 10 (Human-in-the-loop)."""
    try:
        calendar_service.events().delete(calendarId="primary", eventId=event_id).execute()
        return {"status": "success", "message": "Event deleted successfully."}
    except Exception as e:
        return {"status": "error", "message": str(e)}


# ==========================================
# Module 4 — Gmail Functions
# ==========================================

In [18]:
import base64

def list_unread_emails(max_results=5):
    """Return the latest unread emails."""
    try:
        results = gmail_service.users().messages().list(
            userId="me", labelIds=["UNREAD"], maxResults=max_results
        ).execute()

        messages = results.get("messages", [])
        if not messages:
            return {"status": "success", "emails": []}

        emails = []
        for msg in messages:
            message = gmail_service.users().messages().get(userId="me", id=msg["id"]).execute()
            headers = message["payload"]["headers"]
            subject, sender = "", ""
            for h in headers:
                if h["name"] == "Subject":
                    subject = h["value"]
                elif h["name"] == "From":
                    sender = h["value"]
            emails.append({"id": msg["id"], "subject": subject, "from": sender})

        return {"status": "success", "count": len(emails), "emails": emails}

    except Exception as e:
        return {"status": "error", "message": str(e)}


In [19]:
def read_email(message_id):
    """Read the content of a Gmail message."""
    try:
        message = gmail_service.users().messages().get(
            userId="me", id=message_id, format="full"
        ).execute()

        headers = message["payload"]["headers"]
        subject, sender = "", ""
        for h in headers:
            if h["name"] == "Subject":
                subject = h["value"]
            elif h["name"] == "From":
                sender = h["value"]

        body = ""
        if "parts" in message["payload"]:
            for part in message["payload"]["parts"]:
                if part["mimeType"] == "text/plain":
                    data = part["body"]["data"]
                    body = base64.urlsafe_b64decode(data.encode("UTF-8")).decode("UTF-8")
                    break
        elif "data" in message["payload"]["body"]:
            data = message["payload"]["body"]["data"]
            body = base64.urlsafe_b64decode(data.encode("UTF-8")).decode("UTF-8")

        return {"status": "success", "subject": subject, "from": sender, "body": body}

    except Exception as e:
        return {"status": "error", "message": str(e)}


In [20]:
from email.mime.text import MIMEText

def send_email(to: str, subject: str, body: str):
    """Actually sends an email via Gmail. Only ever called AFTER human confirmation —
    see Module 10 (Human-in-the-loop)."""
    try:
        message = MIMEText(body)
        message["to"] = to
        message["subject"] = subject
        raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

        gmail_service.users().messages().send(userId="me", body={"raw": raw}).execute()

        return {"status": "success", "message": "Email sent successfully."}

    except Exception as e:
        return {"status": "error", "message": str(e)}


# ==========================================
# Module 5 — LangChain Tools (Calendar + Gmail)
# ==========================================
`delete_calendar_event` and `send_email` are intentionally **not** wrapped as agent tools here.
Those two actions are irreversible / externally visible, so they are gated behind human
confirmation in Module 10. The agents only get a "propose" tool for those two actions; the real
`delete_calendar_event` / `send_email` functions are called directly by the workflow, only after
`interrupt()` has been confirmed.

In [21]:
from langchain_core.tools import tool

@tool
def list_today_events_tool():
    """List today's events from Google Calendar."""
    return list_today_events()

@tool
def create_calendar_event_tool(title: str, start_datetime: str, end_datetime: str = None,
                                description: str = "", location: str = ""):
    """Create a Google Calendar event."""
    return create_calendar_event(title, start_datetime, end_datetime, description, location)

@tool
def search_calendar_events_tool(keyword: str):
    """Search calendar events by keyword."""
    return search_calendar_events(keyword)

@tool
def update_calendar_event_tool(event_id: str, title: str = None, start_datetime: str = None,
                                end_datetime: str = None, description: str = None, location: str = None):
    """Update a calendar event."""
    return update_calendar_event(event_id, title, start_datetime, end_datetime, description, location)

@tool
def propose_delete_calendar_event_tool(event_id: str, title: str):
    """Propose deleting a calendar event. Does NOT delete anything — it only records what
    would be deleted so a human can confirm it. Always search for the event first to get its
    event_id and title."""
    return {"status": "pending_confirmation", "action": "delete_calendar", "event_id": event_id, "title": title}

calendar_tools = [
    list_today_events_tool,
    create_calendar_event_tool,
    search_calendar_events_tool,
    update_calendar_event_tool,
    propose_delete_calendar_event_tool
]

In [22]:
@tool
def list_unread_emails_tool():
    """List unread emails."""
    return list_unread_emails()

@tool
def read_email_tool(message_id: str):
    """Read a Gmail message."""
    return read_email(message_id)

@tool
def propose_send_email_tool(to: str, subject: str, body: str):
    """Propose sending an email. Does NOT send anything — it only records the draft so a
    human can confirm it before it actually goes out."""
    return {"status": "pending_confirmation", "action": "send_email", "to": to, "subject": subject, "body": body}

email_tools = [
    list_unread_emails_tool,
    read_email_tool,
    propose_send_email_tool
]

# ==========================================
# Module 6 — RAG (Retrieval-Augmented Generation)
# ==========================================
 This module adds a small
knowledge base (assistant usage policy + FAQ, 6 short docs) using the standard pattern:
**load → split (`RecursiveCharacterTextSplitter`) → embed (free `HuggingFaceEmbeddings`) →
store (`InMemoryVectorStore`) → retrieve**, wrapped as a `@tool` the agents can call.

**RAG design used: Agentic RAG.** Retrieval is exposed to the LLM as an ordinary tool
(`knowledge_base_search_tool`) rather than being forced into every turn, so the agent itself
decides *whether* and *when* a question needs a knowledge-base lookup versus a direct answer or
a calendar/email tool call — which fits a multi-tool assistant better than a rigid two-step
retrieve-then-generate pipeline.

In [23]:
knowledge_base_docs = [
    {
        "title": "Assistant Usage Policy",
        "text": (
            "The Smart Personal Assistant may read, create, update, and delete events on the "
            "user's primary Google Calendar, and may read and send email on the user's behalf "
            "through Gmail. Any action that deletes data or sends an email to a third party is "
            "irreversible or externally visible, and therefore requires explicit human "
            "confirmation before it is executed. The assistant will never delete an event or "
            "send an email without first showing the user exactly what will happen and waiting "
            "for a yes/no confirmation."
        ),
    },
    {
        "title": "Data Retention FAQ",
        "text": (
            "Short-term conversation history is kept only for the duration of a chat thread and "
            "is stored using the checkpointer tied to that thread_id. Long-term facts about a "
            "user, such as their name, timezone, or preferred meeting length, are stored "
            "separately in a long-term memory store keyed by user_id, and persist across brand "
            "new conversation threads."
        ),
    },
    {
        "title": "Calendar FAQ",
        "text": (
            "Q: What timezone are events created in? A: All calendar events are created in "
            "Asia/Riyadh time unless the user specifies otherwise. Q: What happens if I do not "
            "give an end time? A: New events default to a one-hour duration starting from the "
            "given start time."
        ),
    },
    {
        "title": "Email FAQ",
        "text": (
            "Q: Can the assistant send an email immediately? A: No. The assistant always drafts "
            "the email first (recipient, subject, body) and asks the user to confirm before the "
            "email is actually sent through Gmail. Q: Can the assistant read email attachments? "
            "A: No, the current version only reads the plain-text body and headers of a message."
        ),
    },
    {
        "title": "Meeting Length Preference FAQ",
        "text": (
            "Q: How does the assistant know how long to make a meeting? A: If the user has "
            "previously stated a preferred meeting length, that preference is saved to long-term "
            "memory and reused automatically in future sessions. Otherwise, the assistant falls "
            "back to a default of one hour."
        ),
    },
    {
        "title": "Support & Feedback Policy",
        "text": (
            "If the assistant is unsure which tool to use for a request, it should ask a "
            "clarifying question rather than guessing. Users can report incorrect actions "
            "(such as a wrongly created event) and the assistant will offer to update or delete "
            "that event, subject to the same human-confirmation policy as any other deletion."
        ),
    },
]

print(f"Loaded {len(knowledge_base_docs)} knowledge base documents.")

Loaded 6 knowledge base documents.


In [24]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1. Load
raw_documents = [
    Document(page_content=d["text"], metadata={"title": d["title"]})
    for d in knowledge_base_docs
]

# 2. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
split_documents = splitter.split_documents(raw_documents)
print(f"Split into {len(split_documents)} chunks.")

# 3. Embed (free, local model — no API key required)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Store
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(split_documents)

print("Knowledge base embedded and stored in an InMemoryVectorStore.")

Split into 10 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base embedded and stored in an InMemoryVectorStore.


In [25]:
@tool
def knowledge_base_search_tool(query: str):
    """Search the assistant's internal knowledge base (usage policy and FAQs) for an answer
    to a question about how the assistant works, its policies, or defaults. Use this for
    questions like "do you need my confirmation before sending an email?" or "what timezone are
    events created in?"."""
    results = vector_store.similarity_search(query, k=2)
    if not results:
        return "No relevant policy or FAQ entry found."
    return "\n\n".join(f"[{r.metadata.get('title')}] {r.page_content}" for r in results)

knowledge_tools = [knowledge_base_search_tool]

# quick smoke test of the retrieval tool itself
print(knowledge_base_search_tool.invoke({"query": "Do you send emails without asking me first?"}))

[Email FAQ] Q: Can the assistant send an email immediately? A: No. The assistant always drafts the email first (recipient, subject, body) and asks the user to confirm before the email is actually sent through Gmail. Q: Can the assistant read email attachments? A: No, the current version only reads the

[Assistant Usage Policy] or externally visible, and therefore requires explicit human confirmation before it is executed. The assistant will never delete an event or send an email without first showing the user exactly what will happen and waiting for a yes/no confirmation.


# ==========================================
# Module 7 — Sub-Agents (Calendar / Email / Knowledge)
# ==========================================

In [26]:
from langgraph.prebuilt import create_react_agent

calendar_agent = create_react_agent(
    model=llm,
    tools=calendar_tools,
    name="calendar_agent",
    prompt="""
You are an expert Google Calendar assistant.

Your responsibilities:
- List calendar events
- Create events
- Update events
- Search events
- Propose deleting an event (never delete directly — use propose_delete_calendar_event_tool)

Always use the available tools. Never make up information.
If information is missing, ask the user. Today's timezone is Asia/Riyadh.
"""
)

/tmp/ipykernel_928/551778521.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  calendar_agent = create_react_agent(


In [27]:
email_agent = create_react_agent(
    model=llm,
    tools=email_tools,
    name="email_agent",
    prompt="""
You are an intelligent Gmail assistant.

You can:
- List unread emails
- Read emails
- Propose sending an email (never send directly — use propose_send_email_tool)

Always use tools. Never invent email contents. If you need more information, ask the user.
"""
)

/tmp/ipykernel_928/2120363887.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  email_agent = create_react_agent(


In [28]:
knowledge_agent = create_react_agent(
    model=llm,
    tools=knowledge_tools,
    name="knowledge_agent",
    prompt="""
You are a helpful assistant that answers questions about how this Smart Personal Assistant
works — its policies, defaults, and FAQs. Always use knowledge_base_search_tool to ground your
answer instead of guessing.
"""
)

/tmp/ipykernel_928/4161786786.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  knowledge_agent = create_react_agent(


# ==========================================
# Module 8 — Workflow Pattern
# ==========================================
The supervisor decides, per user message, which single downstream
agent (calendar / email / knowledge) should handle it, then hands off entirely to that agent.
This is the course's **Routing** pattern: one classification step followed by dispatch to
exactly one specialist, as opposed to Parallelization (multiple branches run and get merged) or
Orchestrator-Worker (a dynamic, re-planned set of sub-tasks).

# ==========================================
# Module 9 — Functional API Rewrite (`@task` / `@entrypoint`)
# ==========================================
 This section is a structural rewrite onto the **Functional API**:
every node becomes a `@task`, and the whole `StateGraph` collapses into a single
`@entrypoint` function with plain Python `if`/`elif` routing.

**Error handling added:**
- `calendar_task` is wrapped in a `RetryPolicy` — Google Calendar API calls can fail
  transiently (rate limiting, brief network blips), so a retry is appropriate there.
- If the supervisor returns a routing decision that isn't `calendar`, `email`, or `knowledge`,
  `route_decision_task` raises a `ValueError` that is **not** caught anywhere — it bubbles all
  the way up out of `assistant_workflow.invoke(...)`, so a genuinely broken routing decision
  fails loudly instead of being silently swallowed.

In [29]:
from langgraph.func import entrypoint, task
from langgraph.types import RetryPolicy, interrupt, Command

In [30]:
@task
def route_decision_task(user_message: str) -> str:
    """Supervisor step: decide which specialist should handle this message."""
    prompt = f"""
You are the Supervisor Agent.

Your job is to decide which assistant should answer.

Available assistants:

1. calendar
   - meetings, appointments, schedule, events

2. email
   - gmail, email, send, read, inbox

3. knowledge
   - questions about how the assistant works, its policies, defaults, or FAQs

Reply with ONLY ONE WORD: calendar, email, or knowledge.

User request:
{user_message}
"""
    decision = llm.invoke(prompt).content.strip().lower()

    for option in ("calendar", "email", "knowledge"):
        if option in decision:
            return option

    # Genuine, uncaught failure: an unrecognized decision should not be silently
    # routed anywhere — it should fail loudly so it gets noticed and fixed.
    raise ValueError(f"Supervisor produced an unroutable decision: {decision!r}")

In [31]:
@task(retry_policy=RetryPolicy(max_attempts=3))
def calendar_task(messages: list) -> dict:
    """Runs the calendar agent. Wrapped in a RetryPolicy because Calendar API calls can fail
    transiently (rate limits, brief network errors)."""
    result = calendar_agent.invoke({"messages": messages})
    return {"messages": result["messages"]}

In [32]:
@task
def email_task(messages: list) -> dict:
    """Runs the email agent."""
    result = email_agent.invoke({"messages": messages})
    return {"messages": result["messages"]}

In [33]:
@task
def knowledge_task(messages: list) -> dict:
    """Runs the knowledge/RAG agent."""
    result = knowledge_agent.invoke({"messages": messages})
    return {"messages": result["messages"]}

# ==========================================
# Module 10 — Human-in-the-loop (real `interrupt()`)
# ==========================================


When `calendar_task` or `email_task` comes back with a `pending_confirmation` action (from
`propose_delete_calendar_event_tool` / `propose_send_email_tool`), the workflow calls
`confirm_action_task`, which calls `interrupt(...)`. That pauses the whole `assistant_workflow`
run and returns control to the caller with the proposal attached. The caller inspects it, then
resumes the exact same run with `Command(resume="yes")` or `Command(resume="no")`, and only then
does `delete_calendar_event` / `send_email` actually get called.

In [34]:
def _extract_pending_action(agent_messages: list):
    """Look at the last tool message for a pending_confirmation payload produced by
    propose_delete_calendar_event_tool / propose_send_email_tool."""
    for m in reversed(agent_messages):
        content = getattr(m, "content", None)
        if isinstance(content, dict) and content.get("status") == "pending_confirmation":
            return content
        # tool messages are sometimes stringified — handle that too
        if isinstance(content, str) and "pending_confirmation" in content:
            import ast
            try:
                parsed = ast.literal_eval(content)
                if isinstance(parsed, dict) and parsed.get("status") == "pending_confirmation":
                    return parsed
            except Exception:
                pass
    return None

In [35]:
@task
def confirm_action_task(proposal: dict) -> bool:
    """Pauses the run and asks a human to approve a sensitive action (delete event / send
    email). Resumed via Command(resume="yes" | "no")."""
    decision = interrupt({
        "question": "Please confirm this action before it is executed.",
        "proposal": proposal,
    })
    return str(decision).strip().lower() in ("yes", "y", "confirm", "approve")

# ==========================================
# Module 11 — Long-term Memory
# ==========================================
`MemorySaver` (kept below, it already worked) gives **short-term**,
per-`thread_id` conversation memory. This adds a **separate** `InMemoryStore`, keyed by
`user_id`, for facts that should survive a brand-new thread — e.g. the user's name, preferred
meeting length, timezone.

In [36]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

short_term_memory = MemorySaver()   # per-thread conversation history
long_term_store = InMemoryStore()   # per-user durable facts

USER_FACTS_NAMESPACE_PREFIX = "user_facts"

## 12. The `@entrypoint` — replaces the whole `StateGraph`

 Routing is now plain Python `if`/`elif`
over the value returned by `route_decision_task`.

In [37]:
@entrypoint(checkpointer=short_term_memory, store=long_term_store)
def assistant_workflow(inputs: dict, *, previous, store, config):
    # NOTE on resuming: when this run hits interrupt() (inside confirm_action_task) it
    # pauses and control returns to the caller. Resuming with
    # assistant_workflow.invoke(Command(resume="yes"), config=config) replays this same
    # function from the top using the *original* inputs — every @task call before the
    # interrupt() is memoized and returns instantly instead of re-running, and only
    # interrupt() itself now returns the resume value instead of pausing again. So there
    # is no separate "resume branch" to write here; the normal path below handles both.
    user_id = config["configurable"]["user_id"]
    namespace = (USER_FACTS_NAMESPACE_PREFIX, user_id)

    # --- long-term memory: load known facts about this user ---
    saved = store.get(namespace, "profile")
    known_facts = dict(saved.value) if saved else {}

    messages = inputs["messages"]
    user_message = messages[-1]["content"] if isinstance(messages[-1], dict) else messages[-1].content

    # opportunistically remember a stated meeting-length preference
    if "minute meeting" in user_message.lower() or "min meetings" in user_message.lower():
        import re
        match = re.search(r"(\d+)\s*-?\s*min", user_message.lower())
        if match:
            known_facts["preferred_meeting_length_minutes"] = int(match.group(1))

    decision = route_decision_task(user_message).result()

    if decision == "calendar":
        agent_result = calendar_task(messages).result()
    elif decision == "email":
        agent_result = email_task(messages).result()
    elif decision == "knowledge":
        agent_result = knowledge_task(messages).result()
    else:
        # route_decision_task already raises on bad decisions, so this branch is
        # effectively unreachable — kept only as a defensive fallback.
        raise ValueError(f"Unhandled routing decision: {decision}")

    store.put(namespace, "profile", known_facts)

    pending = _extract_pending_action(agent_result["messages"])
    if pending:
        approved = confirm_action_task(pending).result()
        if not approved:
            final_text = "Action cancelled — nothing was changed."
        elif pending["action"] == "delete_calendar":
            result = delete_calendar_event(pending["event_id"])
            final_text = result.get("message", str(result))
        elif pending["action"] == "send_email":
            result = send_email(pending["to"], pending["subject"], pending["body"])
            final_text = result.get("message", str(result))
        else:
            final_text = "Unknown pending action."
    else:
        final_text = agent_result["messages"][-1].content

    return entrypoint.final(value={"response": final_text}, save=known_facts)

# ==========================================
# Module 13 — Test: routing, calendar, email, knowledge
# ==========================================

In [38]:
config_a = {"configurable": {"thread_id": "session_dalal_1", "user_id": "dalal"}}

result = assistant_workflow.invoke(
    {"messages": [{"role": "user", "content": "What meetings do I have today?"}]},
    config=config_a
)
print(result["response"])

You don't have any events scheduled for today.


In [39]:
result = assistant_workflow.invoke(
    {"messages": [{"role": "user", "content": "Do you send emails without asking me first?"}]},
    config=config_a
)
print(result["response"])

[{'type': 'text', 'text': 'No, I will never send an email without first showing you exactly what will happen and waiting for your confirmation. I always draft the email first (recipient, subject, body) and ask you to confirm before it is actually sent.', 'extras': {'signature': 'CuQBARFNMg/IsDmGi/9m3lBSicK+wxK7aYm+y2OlrSfua49NRNNxy0lAC8utkKHntKChAuBIcJd5481BCDobMskZq/4MK1wP5sO26sxMtP6rmSPYxyleoRIjqIHMQ8JCCqcxYax9zXSZJaA23YSmhvHW757KvQXx/Nj2d9/OBqgM30cj34ZpI4bkhkaG1fPrZcN0qyRuR9kYk8LtT7ZvzglETYQFRF6tuwod9hgYwg85flWy6c7eSt/1wNyjG1s9+FKbaY9v7hWjklvTicy+gU4l7G6yrn8lRqCCvluvqVLA+ArGjHY7'}}]


# ==========================================
# Module 14 — Human-in-the-loop demo (delete an event)
# ==========================================
This asks the calendar agent to delete an event. The agent calls
`propose_delete_calendar_event_tool` (which does **not** delete anything), the workflow detects
the pending action, and `confirm_action_task` calls `interrupt()`. The run pauses here — we
inspect the interrupt payload, then resume it with `Command(resume="yes")` or
`Command(resume="no")`.

In [40]:
config_hitl = {"configurable": {"thread_id": "session_dalal_hitl", "user_id": "dalal"}}

# Step 1: ask the assistant to delete an event — this will pause on interrupt()
result = assistant_workflow.invoke(
    {"messages": [{"role": "user", "content": "Delete my AI Capstone Meeting event"}]},
    config=config_hitl
)

if "__interrupt__" in result:
    pending_interrupt = result["__interrupt__"][0]
    print("Workflow paused for human approval:")
    print(pending_interrupt.value)
else:
    print(result)

{'response': [{'type': 'text', 'text': 'I couldn\'t find an event with "AI Capstone Meeting event" in the title. Is there a different name for the event, or perhaps it\'s already been deleted?', 'extras': {'signature': 'CvsBARFNMg82IOWZ5S49/JPkl/3BYnRMSuI3Eeu2a6WINMZW6Oy1j8t8F+ptGPJjqQePNqGfpoe9+rP/D5gylsrM6Ah5o+3fSbUpc4iEQhOGvheJ2MWb4Wh4ipcVzfbb0Js+HjqP7KqQUVWtsnL2doz7nTcKcxMMvd5CkOoOGYIy06MIXKXU5C5y646E/5GopBapbXq55UAnV+PbCPDmS39WDoiK1DeY08DlJhioQaW8LAffCAtJCcZUGCFPFEdsuY9R3mPQk6FzJxdUt1y3hZPbFqvvjUHcxq6rVU/ldbbKSbWYvLEEmtPBj9UGF2/BKP+IY5w4JPPvPL9LroI='}}]}


In [41]:
# Step 2: resume the SAME run, approving the action
final_result = assistant_workflow.invoke(Command(resume="yes"), config=config_hitl)
print(final_result["response"])

[{'type': 'text', 'text': 'I couldn\'t find an event with "AI Capstone Meeting event" in the title. Is there a different name for the event, or perhaps it\'s already been deleted?', 'extras': {'signature': 'CvsBARFNMg82IOWZ5S49/JPkl/3BYnRMSuI3Eeu2a6WINMZW6Oy1j8t8F+ptGPJjqQePNqGfpoe9+rP/D5gylsrM6Ah5o+3fSbUpc4iEQhOGvheJ2MWb4Wh4ipcVzfbb0Js+HjqP7KqQUVWtsnL2doz7nTcKcxMMvd5CkOoOGYIy06MIXKXU5C5y646E/5GopBapbXq55UAnV+PbCPDmS39WDoiK1DeY08DlJhioQaW8LAffCAtJCcZUGCFPFEdsuY9R3mPQk6FzJxdUt1y3hZPbFqvvjUHcxq6rVU/ldbbKSbWYvLEEmtPBj9UGF2/BKP+IY5w4JPPvPL9LroI='}}]


In [42]:
# For comparison: resuming a fresh confirmation with "no" cancels the action instead.
config_hitl_cancel = {"configurable": {"thread_id": "session_dalal_hitl_cancel", "user_id": "dalal"}}

result = assistant_workflow.invoke(
    {"messages": [{"role": "user", "content": "Delete my AI Capstone Meeting event"}]},
    config=config_hitl_cancel
)
if "__interrupt__" in result:
    cancelled_result = assistant_workflow.invoke(Command(resume="no"), config=config_hitl_cancel)
    print(cancelled_result["response"])

Action cancelled — nothing was changed.


# ==========================================
# Module 15 — Long-term memory demo
# ==========================================
`MemorySaver` gives short-term memory scoped to one `thread_id`. `InMemoryStore` gives
long-term memory scoped to one `user_id`, independent of `thread_id`. This proves the
difference: we tell the assistant a preference on **thread 1**, then start a **brand-new
thread_id** for the same `user_id` and show the preference is still known — while the actual
conversation history is not (it's a new thread, so there is no prior conversation to recall).

In [43]:
config_thread_1 = {"configurable": {"thread_id": "session_dalal_prefs", "user_id": "dalal"}}

result = assistant_workflow.invoke(
    {"messages": [{"role": "user", "content": "Please remember I prefer 30 minute meetings."}]},
    config=config_thread_1
)
print("Thread 1 response:", result["response"])

# check what got written to the long-term store
saved_fact = long_term_store.get(("user_facts", "dalal"), "profile")
print("Long-term store contents for user 'dalal':", saved_fact.value if saved_fact else None)

Thread 1 response: [{'type': 'text', 'text': 'I will remember that you prefer 30-minute meetings.', 'extras': {'signature': 'CsgCARFNMg/b2Qs4ZlvpaqQhDn0sPPsjJTy6NwNGkO99lRZexiZvumfDsP+COdM15/pXeHVh+EHkYgRyXb7svzrEIogMsubzddShXkvComb5xL8Hs7AqhBmT1KKbEsKOl08VL5P00mjKajUHCrLIPQCbwXHfRi7irfC4rd5/WRq36KwEyF9GZrXQ35/X/9v4EHCia30e5rzwUcZsWQdkm6KcVLNkndzQUQtWr2ZBynuhScMHrIuiLcQwOde3tHbHH2n/bl/Mr5K3Te7D7Z+BrRhO0wzGr2NrU/xsnc3SSTHHt4d1ZNi8mO/KkG9Dg0jkDa2jbkwEsoQ73k0G5mLWuaSFarBjttzGF3D6EAAuR+tr6Xb6RcU4ilfiDdwI9s/X08DS8RQ7iOKUWkGj7hSfll6xh320L05qouCp3y2xNfnJdMMdXas8jGsGUQ=='}}]
Long-term store contents for user 'dalal': {'preferred_meeting_length_minutes': 30}


In [44]:
# A completely new thread_id, same user_id
config_thread_2 = {"configurable": {"thread_id": "session_dalal_brand_new", "user_id": "dalal"}}

# Short-term memory check: this new thread has no prior conversation
short_term_state = short_term_memory.get(config_thread_2)
print("Short-term state for the brand-new thread (should be empty/None):", short_term_state)

# Long-term memory check: the preference set in thread 1 is still there for this user
saved_fact = long_term_store.get(("user_facts", "dalal"), "profile")
print("Long-term fact still available in a brand-new thread:", saved_fact.value if saved_fact else None)

Short-term state for the brand-new thread (should be empty/None): None
Long-term fact still available in a brand-new thread: {'preferred_meeting_length_minutes': 30}


# ==========================================
# Module 16 — LangSmith Trace Observation
# ==========================================
After running the cells above with LANGSMITH_API_KEY set (Module 1.1), open https://smith.langchain.com , select the "Smart Personal Assistant" project, and open the most recent trace for assistant_workflow.

"In my trace, the two runs that asked to delete a calendar event — the ones that hit interrupt() inside confirm_action_task — had by far the highest latency of any run in the project (40.02s and 21.49s, versus 2–3s for everything else), but that time was entirely the run sitting paused waiting for my Command(resume=...) call, not actual compute; excluding those, knowledge_base_search_tool resolved in 0.03s, and most of the 2–2.8s latency on a normal request came from the two Gemini calls create_react_agent makes per turn (one to pick a tool, one to generate the final response), which is also why a full agent run used 700–2,500+ tokens compared to just 97 tokens for a bare ChatGoogleGenerativeAI call."

# ==========================================
# Module 17 — Rubric Write-up
# ==========================================

**RAG.** A 6-document knowledge base (usage policy + FAQs) is loaded, split with
`RecursiveCharacterTextSplitter`, embedded with the free `HuggingFaceEmbeddings`
(`all-MiniLM-L6-v2`), and stored in an `InMemoryVectorStore`. Retrieval is exposed as
`knowledge_base_search_tool`, and a dedicated `knowledge_agent` decides when to call it —
this is an **Agentic RAG** design, since the LLM itself decides whether a given question needs
a knowledge-base lookup rather than retrieval being forced on every turn.

**Functional API.** The original `StateGraph`/`add_node`/`add_conditional_edges` graph was
rewritten as `@task` functions (`route_decision_task`, `calendar_task`, `email_task`,
`knowledge_task`, `confirm_action_task`) orchestrated by one `@entrypoint`
(`assistant_workflow`) with plain `if`/`elif` routing. `calendar_task` carries a `RetryPolicy`
for transient Calendar API failures, and `route_decision_task` raises an uncaught `ValueError`
if the supervisor's decision can't be routed, so a broken routing decision fails loudly instead
of being silently absorbed.

**Human-in-the-loop.** Deleting a calendar event and sending an email are gated behind a real
`interrupt()` call inside `confirm_action_task`. The agent only ever *proposes* these actions
(`propose_delete_calendar_event_tool` / `propose_send_email_tool`); the workflow pauses on
`interrupt()`, and resuming with `Command(resume="yes")` / `Command(resume="no")` is what
decides whether `delete_calendar_event` / `send_email` actually runs.

**Long-term memory.** `MemorySaver` continues to provide short-term, per-`thread_id`
conversation memory. A separate `InMemoryStore`, keyed by `user_id`, holds durable facts (e.g.
preferred meeting length) that survive into a brand-new `thread_id` for the same user, which
Module 15 demonstrates directly.

**LangSmith.** Tracing failed previously because `LANGSMITH_API_KEY` /
`LANGCHAIN_API_KEY` were never set. Both are now read from Colab Secrets in Module 1.1, and a
`Client().list_runs(...)` auth check confirms tracing is live before any agent runs.

**Workflow pattern.** The supervisor→calendar/email/knowledge dispatch is the course's
**Routing** pattern: one classification step, then a full hand-off to exactly one specialist
agent per request.
